# probando búsquedas en la base

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()


True

elijo la base

In [3]:
from pymongo import MongoClient
import os

uri = os.getenv("URI_mia")
client = MongoClient(uri)

db = client["catalogo_repuestos"]        
coleccion = db["repuestos_internos"]

elegir modelos para embeddings

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/home/nicolas/entornos/trabajo-final-modulo6_python3.11.13/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## búsqueda semántica PURA

función para búsquedas semánticas

In [ ]:
def buscar_semantico(texto, n=5):
    # 1. Vectorizamos la consulta
    query_vector = embedder.embed_query(texto)

    # 2. Ejecutamos el vectorSearch
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",      
                "path": "embedding",
                "queryVector": query_vector,
                "numCandidates": 100,
                "limit": n
            }
        }
    ]

    resultados = coleccion.aggregate(pipeline)
    return list(resultados)


: 

: 

: 

las búsquedas semánticas

In [ ]:
res = buscar_semantico("pastillas de freno", n=5)

for r in res:
    print(r["Descripción"], "-", r["Marca"], "- (", r["Marca Vehículo"], r["Modelo"], r["Año"],") - $", r.get("Precio"))
    
#print(res)

Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694
Pastillas de freno - Ferodo - ( Ford Focus 2006-2012 ) - $ 110840
Pastillas de freno - Brembo - ( Chevrolet Cruze 2006-2012 ) - $ 115248
Pastillas de freno - Brembo - ( Fiat Uno 2020-2024 ) - $ 190180
Pastillas de freno - Ferodo - ( Toyota Hilux 2000-2005 ) - $ 56204


: 

: 

: 

In [ ]:
res = buscar_semantico("Marca: Toyota; Modelo: Etios; año: 2009; Descripción: pastillas de freno", n=5)
for r in res:
    print(r["Descripción"], "-", r["Marca"], "- (", r["Marca Vehículo"], r["Modelo"], r["Año"],") - $", r.get("Precio"))
    
#print(res)

Pastillas de freno - TRW - ( Toyota Corolla 2006-2012 ) - $ 57096
Pastillas de freno - Ferodo - ( Toyota Hilux 2000-2005 ) - $ 56204
Amortiguador delantero - Sachs - ( Toyota Corolla 2000-2005 ) - $ 91018
Pastillas de freno - Ferodo - ( Chevrolet Corsa 2020-2024 ) - $ 199914
Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694


: 

: 

: 

: 

: 

: 

## Busqueda semática mixta (con filtros *duros*)

Ahora vamos por meter fitros **duros** en el query

In [5]:
def buscar_semantico_filtrado(marca_vehiculo, modelo, anio, descripcion, n=5):
    # 1) Vectorizamos SOLO la descripción
    query_vector = embedder.embed_query(descripcion)

    # 2) Armamos la query con filtro estructurado
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "path": "embedding",
                "queryVector": query_vector,
                "numCandidates": 100,
                "limit": n,
                "filter": {
                    "Marca Vehículo": marca_vehiculo,
                    "Modelo": modelo#,
                    # si Año es texto tipo "2006-2012", jugamos con regex
                    #"Año": {"$regex": str(anio)}
                }
            }
        },
        ## entiendo que esta parte es opcional,
        # Acá se genera "el documento que se devuelve"... podríamos omitir
        # campos (por en este caso _id)
        {
            "$project": {
                "_id": 0,
                "Descripción": 1,
                "Marca": 1,
                "Marca Vehículo": 1,
                "Modelo": 1,
                "Año": 1,
                "Precio": 1,
                "Stock": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]

    return list(coleccion.aggregate(pipeline))


filtro para evitar "falsos positivos"

In [6]:
def filtrar_por_score_relativo(resultados, min_abs=0.6, factor_rel=0.9):
    """
    - min_abs: score mínimo absoluto. Por debajo de este valor, se descarta.
    - factor_rel: porcentaje del mejor score (0.9 = 90% del mejor). Los que estén por debajo de este umbral se descartan.
    Retorna la lista filtrada de resultados.
    """
    if not resultados:
        return []

    top = resultados[0]["score"]
    filtrados = []
    for r in resultados:
        if r["score"] >= min_abs and r["score"] >= top * factor_rel:
            filtrados.append(r)
    return filtrados


lo probamos

In [7]:
res = buscar_semantico_filtrado(
    marca_vehiculo="Toyota",
    modelo="Etios",
    anio=2009,
    descripcion="pastillas",
    n=5
)

for r in res:
    print(
        r["Descripción"],
        "-",
        r["Marca"],
        "- (", r["Marca Vehículo"], r["Modelo"], r["Año"], ") - $", r["Precio"],
        "(",r["Stock"], "unidades) | score:", r["score"]
    )


Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694 ( 78 unidades) | score: 0.7938084602355957
Pastillas de freno - Ferodo - ( Toyota Etios 2024-2025 ) - $ 32000 ( 12 unidades) | score: 0.713097095489502
Bobina de encendido - Valeo - ( Toyota Etios 2006-2012 ) - $ 115078 ( 5 unidades) | score: 0.654833197593689
Bobina de encendido - Bosch - ( Toyota Etios 2020-2024 ) - $ 181471 ( 51 unidades) | score: 0.6473424434661865
Alternador - Valeo - ( Toyota Etios 2013-2020 ) - $ 112984 ( 72 unidades) | score: 0.6346254944801331


In [7]:
resultados_filtrados = filtrar_por_score_relativo(res, min_abs=0.6, factor_rel=0.9)
for r in resultados_filtrados:
    print(
        r["Descripción"],
        "-",
        r["Marca"],
        "- (", r["Marca Vehículo"], r["Modelo"], r["Año"], ") - $", r["Precio"],
        "(",r["Stock"], "unidades) | score:", r["score"]
    )

Bomba de freno - Ferodo - ( Toyota Etios 2024-2025 ) - $ 72000 ( 10 unidades) | score: 0.7889045476913452
Pastillas de freno - Brembo - ( Toyota Etios 2006-2012 ) - $ 123694 ( 78 unidades) | score: 0.787067711353302
Bobina de encendido - Valeo - ( Toyota Etios 2006-2012 ) - $ 115078 ( 5 unidades) | score: 0.7226150631904602
Pastillas de freno - Ferodo - ( Toyota Etios 2024-2025 ) - $ 32000 ( 12 unidades) | score: 0.720238447189331
